# 🔎 Embeddings & Search: Zero to Hero — A Guided Lab

Build a search engine **from scratch** — keyword search (TF-IDF, BM25), semantic search
(embeddings + cosine similarity), and hybrid search — then evaluate quality with real metrics.

**Runs 100% offline** using NumPy and small text corpora. The concepts map directly to real
embedding models (OpenAI, Sentence-Transformers) and vector databases.

**How this lab works** — 📖 Theory → 🧠 Mental model → 🖼️ ASCII diagram → 🔬 Worked example →
⚡ Pro tips → ⚠️ Traps → ✏️ Your Turn → ✅ Solution.

**Roadmap**
1. The search problem & our corpus
2. Tokenization & the bag-of-words vector
3. Cosine similarity
4. TF-IDF (smart keyword weighting)
5. BM25 (the industry keyword standard)
6. Embeddings & semantic search (the idea)
7. Hybrid search (best of both)
8. Chunking documents
9. Evaluating search (precision@k, recall@k, MRR)
10. 🏆 Capstone: a complete mini search engine


In [ ]:
import numpy as np
import re
from collections import Counter

# Our corpus: customer-support knowledge base
corpus = [
    "How do I reset my forgotten password?",
    "The mobile app keeps crashing on startup",
    "I want a refund for my recent order",
    "How can I change my billing address?",
    "The application is very slow to load",
    "I forgot my login credentials and cannot sign in",
    "Can I get money back for a cancelled subscription?",
    "The camera feature freezes the whole app",
    "How do I update my payment method?",
    "My package has not arrived yet",
]
print(f"{len(corpus)} documents in corpus")

---
## Chapter 1 — The Search Problem

📖 **Theory.** Search = given a **query**, rank **documents** by relevance. The whole game is
defining "relevance" as a number you can compute and sort by. Two families:
- **Keyword search** (TF-IDF, BM25): matches shared *words*. Great for exact terms.
- **Semantic search** (embeddings): matches *meaning*. Catches paraphrases/synonyms.

🖼️ **Diagram — the search pipeline**
```
 query ──► [ score against each doc ] ──► sort by score ──► top-k results
              (TF-IDF / BM25 / embeddings)
```

🧠 **Mental model.** Every search method is just a different **scoring function** `score(query,
doc)`. Everything else (indexing, ranking) is plumbing around that score.


In [ ]:
def tokenize(text):
    """lowercase + split into word tokens"""
    return re.findall(r"[a-z]+", text.lower())

print(tokenize("How do I RESET my password?"))
# vocabulary across the whole corpus
vocab = sorted(set(w for doc in corpus for w in tokenize(doc)))
print(f"\nvocabulary size: {len(vocab)}")
print("sample words:", vocab[:10])

### ✏️ Your Turn 1.1
Write `doc_frequency(word)` that counts how many documents in `corpus` contain a given word
(at least once). Test it on `"app"` and `"password"`.

In [ ]:
def doc_frequency(word):
    pass
print(doc_frequency("app"), doc_frequency("password"))

✅ **Solution**
```python
def doc_frequency(word):
    return sum(1 for doc in corpus if word in tokenize(doc))
```

---
## Chapter 2 — Tokenization & Bag-of-Words

📖 **Theory.** To do math on text we turn each document into a **vector**. The simplest is
**bag-of-words (BoW)**: a vector as long as the vocabulary, where each position counts how many
times that word appears. Word order is lost (hence "bag").

🖼️ **Diagram — bag-of-words**
```
 vocab:  [app, crash, password, refund, ...]
 "app keeps crashing" -> [1, 0, 0, 0, ...]   (counts per vocab word)
 "refund my refund"   -> [0, 0, 0, 2, ...]
```


In [ ]:
vocab_index = {w: i for i, w in enumerate(vocab)}

def bag_of_words(text):
    vec = np.zeros(len(vocab))
    for w in tokenize(text):
        if w in vocab_index:
            vec[vocab_index[w]] += 1
    return vec

# build a document-term matrix (rows = docs, cols = vocab words)
doc_vectors = np.array([bag_of_words(doc) for doc in corpus])
print("doc-term matrix shape:", doc_vectors.shape)
print("nonzero entries in doc 0:", int((doc_vectors[0] > 0).sum()))

⚠️ **Common trap.** BoW ignores word order and meaning: "dog bites man" and "man bites dog"
get identical vectors. That's why we layer smarter methods on top.

### ✏️ Your Turn 2.1
Using `bag_of_words`, how many *distinct* vocabulary words does the query "app crash refund"
contain that also appear in the vocabulary? (count nonzero positions)

In [ ]:
q_vec = None
distinct_in_vocab = None
print(distinct_in_vocab)

✅ **Solution**
```python
q_vec = bag_of_words("app crash refund")
distinct_in_vocab = int((q_vec > 0).sum())
```

---
## Chapter 3 — Cosine Similarity

📖 **Theory.** To compare two vectors we use **cosine similarity**: the cosine of the angle
between them. It ignores magnitude (document length) and focuses on *direction* (which words,
in what proportion). Range: 0 (no shared direction) to 1 (identical direction).

`cos(a, b) = (a · b) / (‖a‖ · ‖b‖)`

🖼️ **Diagram — angle, not length**
```
   b   a       small angle  -> cos ≈ 1 (similar)
    \\ /
     •         large angle  -> cos ≈ 0 (different)
```

🧠 **Mental model.** Cosine asks "do these point the same way?" not "are they the same size?" —
so a short and long document about the same topic still score high.


In [ ]:
def cosine_similarity(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return 0.0 if denom == 0 else float(np.dot(a, b) / denom)

def search_bow(query, k=3):
    q = bag_of_words(query)
    scores = [(doc, cosine_similarity(q, dv)) for doc, dv in zip(corpus, doc_vectors)]
    return sorted(scores, key=lambda x: -x[1])[:k]

for doc, score in search_bow("my password does not work"):
    print(f"{score:.3f}  {doc}")

### ✏️ Your Turn 3.1
Search for `"refund my money"`. Look at the top results — does BoW find the "money back" doc?
Explain (in a comment) why keyword methods can miss semantically-related wording.

In [ ]:
# run search_bow("refund my money") and observe


✅ **Solution**
```python
for doc, s in search_bow("refund my money"):
    print(round(s,3), doc)
# BoW only matches exact shared words, so "money back" may rank low
# unless the query literally shares words with it.
```

---
## Chapter 4 — TF-IDF (smart keyword weighting)

📖 **Theory.** Raw counts over-weight common words ("the", "my"). **TF-IDF** fixes this:
- **TF** (term frequency): how often a word appears in *this* doc.
- **IDF** (inverse document frequency): how *rare* the word is across all docs — rare words are
  more informative.
- score = TF × IDF. Common words get near-zero IDF; distinctive words dominate.

🖼️ **Diagram — IDF down-weights common words**
```
 word     appears in docs   IDF (rarity)
 "my"        8/10            low   ─┐ down-weighted
 "the"       6/10            low   ─┘
 "refund"    2/10            high  ─── boosted (distinctive)
```


In [ ]:
N = len(corpus)
doc_freq = Counter()
for doc in corpus:
    for w in set(tokenize(doc)):
        doc_freq[w] += 1

def tfidf_vector(text):
    vec = np.zeros(len(vocab))
    tf = Counter(tokenize(text))
    for w, count in tf.items():
        if w in vocab_index:
            idf = np.log((1 + N) / (1 + doc_freq[w])) + 1   # smoothed IDF
            vec[vocab_index[w]] = count * idf
    return vec

tfidf_matrix = np.array([tfidf_vector(doc) for doc in corpus])

def search_tfidf(query, k=3):
    q = tfidf_vector(query)
    scores = [(doc, cosine_similarity(q, dv)) for doc, dv in zip(corpus, tfidf_matrix)]
    return sorted(scores, key=lambda x: -x[1])[:k]

for doc, score in search_tfidf("change payment method"):
    print(f"{score:.3f}  {doc}")

⚡ **Pro tip.** TF-IDF is a strong, fast, interpretable baseline. Many production systems
*start* here and only add embeddings when semantic matching is truly needed.

### ✏️ Your Turn 4.1
Compute the IDF of a **common** word (like "my") and a **rare** word (like "camera"). Confirm
the rare word has higher IDF.

In [ ]:
idf_my = None
idf_camera = None
print(idf_my, idf_camera)

✅ **Solution**
```python
idf_my = np.log((1+N)/(1+doc_freq["my"])) + 1
idf_camera = np.log((1+N)/(1+doc_freq["camera"])) + 1
# idf_camera > idf_my
```

---
## Chapter 5 — BM25 (the industry keyword standard)

📖 **Theory.** **BM25** improves on TF-IDF with two ideas:
1. **Term-frequency saturation** — the 10th occurrence of a word adds less than the 2nd (via
   parameter `k1`).
2. **Length normalization** — long docs don't unfairly win just by being long (parameter `b`).

It's the default ranking in Elasticsearch/Lucene and still competitive with neural methods for
keyword-heavy queries.

🖼️ **Diagram — TF saturation**
```
 score
   │        ___________  BM25 (saturates)
   │      /
   │    / ______________ raw TF (linear, unbounded)
   │  //
   └────────────────► term frequency
```


In [ ]:
class BM25:
    def __init__(self, corpus_tokens, k1=1.5, b=0.75):
        self.docs = corpus_tokens
        self.k1, self.b = k1, b
        self.N = len(corpus_tokens)
        self.avgdl = np.mean([len(d) for d in corpus_tokens])
        self.df = Counter()
        for d in corpus_tokens:
            for w in set(d):
                self.df[w] += 1
    def idf(self, term):
        return np.log((self.N - self.df[term] + 0.5) / (self.df[term] + 0.5) + 1)
    def score(self, query_tokens, doc_idx):
        doc = self.docs[doc_idx]
        dl = len(doc)
        freqs = Counter(doc)
        s = 0.0
        for term in query_tokens:
            if term not in freqs: continue
            f = freqs[term]
            num = f * (self.k1 + 1)
            den = f + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
            s += self.idf(term) * num / den
        return s
    def search(self, query, k=3):
        qt = tokenize(query)
        scores = [(corpus[i], self.score(qt, i)) for i in range(self.N)]
        return sorted(scores, key=lambda x: -x[1])[:k]

corpus_tokens = [tokenize(doc) for doc in corpus]
bm25 = BM25(corpus_tokens)
for doc, score in bm25.search("app crashes on startup"):
    print(f"{score:.3f}  {doc}")

⚡ **Pro tip.** Defaults `k1=1.5, b=0.75` are battle-tested — start there and only tune if
you have an eval set showing they help.

### ✏️ Your Turn 5.1
Run BM25 for the query `"forgot password"` and compare its top result to `search_tfidf` on the
same query. Do they agree on the #1 doc?

In [ ]:
# compare bm25.search("forgot password") vs search_tfidf("forgot password")


✅ **Solution**
```python
print("BM25:", bm25.search("forgot password")[0])
print("TFIDF:", search_tfidf("forgot password")[0])
# Both should surface the password/credentials docs at the top.
```

---
## Chapter 6 — Embeddings & Semantic Search

📖 **Theory.** An **embedding** maps text to a dense vector where *meaning-similar* text is
*geometrically close* — even with **zero shared words**. A real model (OpenAI,
Sentence-Transformers) learns this from billions of examples. We'll **simulate** one with a tiny
hand-built semantic map so the lab stays offline, but the search logic is identical to real usage.

🖼️ **Diagram — semantic space**
```
        "refund" • • "money back"      ← close despite no shared words
                       
   "password" • • "login credentials"
                       
              "camera" •     ← far from the others
```

🧠 **Mental model.** Keyword search matches *spellings*; embeddings match *ideas*. With real
embeddings, "money back" finds the "refund" doc automatically.


In [ ]:
# --- Simulated embeddings: group synonyms into shared "concept" dimensions ---
# (In production: emb = openai.embeddings.create(...) or SentenceTransformer(...).encode(...))
concepts = {
    "password": ["password","forgot","reset","login","credentials","sign"],
    "refund":   ["refund","money","back","cancelled","subscription","order"],
    "crash":    ["crash","crashing","freezes","slow","startup","load","app","application","camera","feature"],
    "billing":  ["billing","payment","address","method","change","update"],
    "shipping": ["package","arrived","yet"],
}
concept_list = list(concepts.keys())
word2concept = {w:i for i,c in enumerate(concept_list) for w in concepts[c]}

def embed(text, dim=len(concept_list)):
    v = np.zeros(dim)
    for w in tokenize(text):
        if w in word2concept:
            v[word2concept[w]] += 1
    # normalize to unit length
    n = np.linalg.norm(v)
    return v/n if n > 0 else v

emb_matrix = np.array([embed(doc) for doc in corpus])

def search_semantic(query, k=3):
    q = embed(query)
    scores = [(doc, cosine_similarity(q, e)) for doc, e in zip(corpus, emb_matrix)]
    return sorted(scores, key=lambda x: -x[1])[:k]

# The magic: "money back" finds the refund doc with NO shared keywords
print("Query: 'get my money back' (note: no word 'refund')")
for doc, score in search_semantic("get my money back"):
    print(f"  {score:.3f}  {doc}")

### ✏️ Your Turn 6.1
Search semantically for `"cannot sign in"` and confirm it retrieves the password/credentials
docs even though the query doesn't contain the word "password".

In [ ]:
# run search_semantic("cannot sign in")


✅ **Solution**
```python
for doc, s in search_semantic("cannot sign in"):
    print(round(s,3), doc)
# The login/credentials/password docs rank top via shared *concept*, not shared words.
```

---
## Chapter 7 — Hybrid Search

📖 **Theory.** Keyword and semantic search have complementary strengths:
- Keyword (BM25): unbeatable for exact terms — product codes, names, error strings.
- Semantic: catches paraphrases/synonyms.
**Hybrid** combines both scores (often a weighted sum after normalizing each to [0,1]). Usually
beats either alone.

🖼️ **Diagram — score fusion**
```
 query ─┬─► BM25 score ───┐
        │                 ├─► α·sem + (1-α)·kw ─► final rank
        └─► semantic ──────┘
```


In [ ]:
def normalize(scores):
    arr = np.array(scores, dtype=float)
    lo, hi = arr.min(), arr.max()
    return (arr - lo) / (hi - lo) if hi > lo else np.zeros_like(arr)

def search_hybrid(query, alpha=0.5, k=3):
    qt = tokenize(query)
    kw = normalize([bm25.score(qt, i) for i in range(len(corpus))])
    qe = embed(query)
    sem = normalize([cosine_similarity(qe, e) for e in emb_matrix])
    combined = alpha * sem + (1 - alpha) * kw
    ranked = sorted(zip(corpus, combined), key=lambda x: -x[1])
    return ranked[:k]

print("Hybrid results for 'refund for cancelled plan':")
for doc, score in search_hybrid("refund for cancelled plan", alpha=0.5):
    print(f"  {score:.3f}  {doc}")

⚡ **Pro tip.** `alpha` is a tuning knob: push toward keyword (low alpha) for
code/ID-heavy search, toward semantic (high alpha) for natural-language questions. Tune it on a
real eval set (Ch.9).

### ✏️ Your Turn 7.1
Run the hybrid search with `alpha=0.0` (pure keyword) and `alpha=1.0` (pure semantic) for the
query `"money back"`. Notice how the ranking shifts.

In [ ]:
# compare alpha=0.0 vs alpha=1.0 for "money back"


✅ **Solution**
```python
print("keyword-only:", search_hybrid("money back", alpha=0.0)[0])
print("semantic-only:", search_hybrid("money back", alpha=1.0)[0])
```

---
## Chapter 8 — Chunking Documents

📖 **Theory.** Real documents are long. You **chunk** them into passages before embedding, so
retrieval returns a focused, relevant snippet (and fits context limits downstream in RAG).
Key knobs:
- **chunk size** — too small loses context; too large dilutes relevance (200–500 words common).
- **overlap** — repeat some words between chunks so answers near a boundary aren't split.

🖼️ **Diagram — sliding window with overlap**
```
 [========= chunk 1 =========]
                     [========= chunk 2 =========]
                                         [======== chunk 3 ...
      |── overlap ──|         |── overlap ──|
```


In [ ]:
def chunk_text(text, chunk_size=10, overlap=3):
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        chunks.append(" ".join(words[start:start+chunk_size]))
        start += chunk_size - overlap
    return chunks

long_doc = ("Our return policy allows returns within thirty days of purchase with a valid "
            "receipt. Refunds are processed within five to seven business days to the "
            "original payment method. Subscriptions can be cancelled anytime from account "
            "settings under the billing tab.")
for i, ch in enumerate(chunk_text(long_doc, chunk_size=12, overlap=4)):
    print(f"chunk {i}: {ch}")

⚠️ **Common trap.** Chunking mid-sentence or mid-table destroys meaning. Prefer
sentence-aware splitting for prose; keep tables/code blocks intact.

### ✏️ Your Turn 8.1
Chunk `long_doc` with `chunk_size=8, overlap=2` and report how many chunks result.

In [ ]:
chunks = None
print(len(chunks) if chunks else None)

✅ **Solution**
```python
chunks = chunk_text(long_doc, chunk_size=8, overlap=2)
print(len(chunks))
```

---
## Chapter 9 — Evaluating Search Quality

📖 **Theory.** "Looks good" isn't a metric. Use a small **labeled set** (query → known relevant
doc) and measure:
- **precision@k** — of the top k results, what fraction are relevant?
- **recall@k** — of all relevant docs, what fraction appear in the top k?
- **MRR** (Mean Reciprocal Rank) — 1/(rank of first correct result), averaged over queries.

🖼️ **Diagram — reciprocal rank**
```
 correct at position 1 -> RR = 1/1 = 1.0
 correct at position 3 -> RR = 1/3 = 0.33
 not in top-k          -> RR = 0
```


In [ ]:
# ground truth: query -> index of the single most relevant doc
ground_truth = {
    "reset my password": 0,
    "app crashing": 1,
    "get money back": 2,
    "update payment method": 8,
    "where is my package": 9,
}

def reciprocal_rank(results, correct_idx):
    for rank, (doc, _) in enumerate(results, start=1):
        if corpus.index(doc) == correct_idx:
            return 1.0 / rank
    return 0.0

def evaluate(search_fn, k=3):
    rrs = []
    for q, correct in ground_truth.items():
        results = search_fn(q, k=k)
        rrs.append(reciprocal_rank(results, correct))
    return np.mean(rrs)

print(f"BM25    MRR@3: {evaluate(lambda q,k: bm25.search(q,k)):.3f}")
print(f"Semantic MRR@3: {evaluate(search_semantic):.3f}")
print(f"Hybrid   MRR@3: {evaluate(lambda q,k: search_hybrid(q, alpha=0.5, k=k)):.3f}")

### ✏️ Your Turn 9.1
Write `precision_at_k(results, correct_idx, k)` returning 1.0 if the correct doc is anywhere in
the top-k results, else 0.0. (For a single relevant doc, this is a hit-rate.)

In [ ]:
def precision_at_k(results, correct_idx, k=3):
    pass


✅ **Solution**
```python
def precision_at_k(results, correct_idx, k=3):
    top = [corpus.index(doc) for doc,_ in results[:k]]
    return 1.0 if correct_idx in top else 0.0
```

---
## 🏆 Chapter 10 — Capstone: A Complete Mini Search Engine

Wrap everything into a `SearchEngine` class that indexes a corpus and supports keyword,
semantic, and hybrid search — then evaluate it. Build it before revealing the solution.

In [ ]:
# Your SearchEngine class here
class SearchEngine:
    def __init__(self, documents):
        pass
    def search(self, query, method="hybrid", alpha=0.5, k=3):
        pass

# engine = SearchEngine(corpus)
# print(engine.search("forgot my login", method="semantic"))


✅ **Capstone Solution**
```python
class SearchEngine:
    def __init__(self, documents):
        self.docs = documents
        self.tokens = [tokenize(d) for d in documents]
        self.bm25 = BM25(self.tokens)
        self.emb = np.array([embed(d) for d in documents])
    def _norm(self, s):
        a = np.array(s, float); lo,hi = a.min(), a.max()
        return (a-lo)/(hi-lo) if hi>lo else np.zeros_like(a)
    def search(self, query, method="hybrid", alpha=0.5, k=3):
        qt = tokenize(query)
        if method == "keyword":
            scores = [self.bm25.score(qt, i) for i in range(len(self.docs))]
        elif method == "semantic":
            qe = embed(query)
            scores = [cosine_similarity(qe, e) for e in self.emb]
        else:  # hybrid
            kw = self._norm([self.bm25.score(qt, i) for i in range(len(self.docs))])
            qe = embed(query)
            sem = self._norm([cosine_similarity(qe, e) for e in self.emb])
            scores = alpha*sem + (1-alpha)*kw
        ranked = sorted(zip(self.docs, scores), key=lambda x: -x[1])
        return ranked[:k]

engine = SearchEngine(corpus)
print(engine.search("forgot my login", method="semantic"))
print(engine.search("refund", method="hybrid", alpha=0.6))
```

🎉 **You built a real search engine from first principles** — tokenization, TF-IDF, BM25,
embeddings, hybrid fusion, chunking, and evaluation. This is exactly the retrieval half of a
RAG system (your next lab!). Swap `embed()` for a real embedding model and `BM25`/vectors for a
vector database, and you have production search.

---
### 📌 Concept Quick-Reference
**Text→vector:** tokenize, bag-of-words, TF-IDF, embeddings
**Similarity:** cosine similarity (angle, ignores length)
**Keyword ranking:** TF-IDF, BM25 (k1 saturation, b length-norm)
**Semantic:** embeddings map meaning→geometry; synonyms score high with no shared words
**Hybrid:** normalize + weighted fuse (alpha) of keyword & semantic
**Chunking:** chunk_size + overlap; sentence-aware for prose
**Evaluation:** precision@k, recall@k, MRR, against a labeled set
